# 01 — Dataset Exploration & Data Preparation (MS COCO 2017)

**Project**: Black-and-White Image Colorization with Deep Learning

This notebook checks the dataset every model trains on:
1. the 12,000-image MS COCO 2017 subset and its 80/10/10 train/validation/test split,
2. the CIELAB preprocessing that turns a color photo into an input (L) and a target (ab),
3. the exploratory data analysis (EDA) plots.

**Before running**: download the images once from the repository root with
`python scripts/prepare_dataset.py --from-splits`.

### Setup

Notebooks live in `notebooks/`, but the code imports the `src` package from the project root.
This cell finds the root (one level up when started from `notebooks/`) and adds it to `sys.path`.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.dataset import ColorizationDataset
from src.utils import lab_normalized_to_rgb

SPLITS_DIR = ROOT / 'data' / 'splits'
IMAGE_DIR = ROOT / 'data' / 'processed_256'
print('Project root:', ROOT)

## 1. Dataset statistics and split integrity

`scripts/prepare_dataset.py` wrote `dataset_stats.json` when the subset was created. It records how many
COCO images were inspected, how many grayscale photos were rejected, and the size of each split.

The three CSV manifests (`train.csv`, `val.csv`, `test.csv`) list the exact images of every split.
The asserts check that no image appears in two splits; otherwise test scores would be inflated by images
the model has already seen.

In [ ]:
with open(SPLITS_DIR / 'dataset_stats.json', 'r', encoding='utf-8') as f:
    stats = json.load(f)
print(json.dumps(stats, indent=2))

splits = {name: pd.read_csv(SPLITS_DIR / f'{name}.csv') for name in ['train', 'val', 'test']}
ids = {name: set(df['id']) for name, df in splits.items()}
assert ids['train'].isdisjoint(ids['val']), 'Overlap between train and val!'
assert ids['train'].isdisjoint(ids['test']), 'Overlap between train and test!'
assert ids['val'].isdisjoint(ids['test']), 'Overlap between val and test!'

total = sum(len(df) for df in splits.values())
augmentation = {'train': 'Random horizontal flip (p=0.5)', 'val': 'None', 'test': 'None'}
display(pd.DataFrame([
    {'Split': name, 'Count': len(df), 'Percentage': f'{len(df) / total * 100:.1f}%', 'Augmentation': augmentation[name]}
    for name, df in splits.items()
]))

## 2. CIELAB preprocessing

`ColorizationDataset` reads a photo, resizes it to 256×256, flips it horizontally at random (training only),
and converts it from RGB to CIELAB:

| Channel | Meaning | Raw range | Normalized | Role |
| --- | --- | --- | --- | --- |
| L | lightness (the grayscale photo) | 0 … 100 | `L / 50 - 1` → [-1, 1] | model input |
| a | green ↔ red | about -128 … 127 | `a / 128` → [-1, 1] | model target |
| b | blue ↔ yellow | about -128 … 127 | `b / 128` → [-1, 1] | model target |

This cell loads one training sample and checks the tensor shapes and value ranges.

In [ ]:
ds = ColorizationDataset(SPLITS_DIR / 'train.csv', IMAGE_DIR, image_size=256, augment=True)

L, ab, file_name = ds[0]
print(f'Sample file : {file_name}')
print(f'L  tensor   : shape={tuple(L.shape)}, dtype={L.dtype}, min={L.min():.4f}, max={L.max():.4f}')
print(f'ab tensor   : shape={tuple(ab.shape)}, dtype={ab.dtype}, min={ab.min():.4f}, max={ab.max():.4f}')

assert L.shape == (1, 256, 256) and ab.shape == (2, 256, 256)
assert -1.01 <= L.min() and L.max() <= 1.01
assert -1.01 <= ab.min() and ab.max() <= 1.01
print('All tensor shape and normalization checks passed!')

### What the model sees and what it must predict

From left to right: the L input, the a and b targets (blue = negative, red = positive), and the RGB image
rebuilt from L + ab with `lab_normalized_to_rgb`, the same function used to score the models.
A model that predicts ab = 0 everywhere would output a gray image, which is why faded colors are the
typical failure of a regression model.

In [ ]:
rgb = lab_normalized_to_rgb(L.unsqueeze(0), ab.unsqueeze(0))[0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
panels = [
    (L[0], 'L (input)', 'gray'),
    (ab[0], 'a (target): green - red', 'coolwarm'),
    (ab[1], 'b (target): blue - yellow', 'coolwarm'),
    (rgb, 'L + ab -> RGB', None),
]
for ax, (image, title, cmap) in zip(axes, panels):
    if cmap == 'coolwarm':
        ax.imshow(image, cmap=cmap, vmin=-0.5, vmax=0.5)
    else:
        ax.imshow(image, cmap=cmap)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Exploratory data analysis

These plots were saved by `scripts/prepare_dataset.py`:
- **CIELAB and resolution distributions**: a and b cluster around 0 (most pixels have low saturation),
  which is why a model trained with MSE tends to predict faded colors.
- **Sample pairs**: grayscale inputs next to their color targets.

In [ ]:
for title, rel_path in [
    ('CIELAB & Resolution Distributions', 'results/plots/dataset_eda.png'),
    ('Sample Input (L) vs Target (RGB) Pairs', 'results/plots/dataset_samples.png'),
]:
    image_path = ROOT / rel_path
    if not image_path.exists():
        print('Missing plot:', image_path)
        continue
    plt.figure(figsize=(14, 7))
    plt.imshow(Image.open(image_path))
    plt.axis('off')
    plt.title(title)
    plt.show()